# 07 - Validación: 200 Tareas de Percepción de Profundidad

**Modelo Cognitivo Artificial para la Replicación de la Actividad Neurofisiológica de Percepción de Profundidad**

- **Autor:** Jesús Goenaga Peña
- **Programa:** Doctorado en Ciencias Cognitivas - Universidad Autónoma de Manizales

---

Según la sección 8.4.2.4 de la tesis, el modelo se somete al **mismo conjunto de
200 tareas** que los participantes humanos. Se realizan **3 ejecuciones completas**
(3 participantes artificiales) para calcular variabilidad interna.

| Bloque | Tareas | Fuente | Tipo |
|--------|--------|--------|------|
| 1-100 | Discriminación de profundidad relativa | KITTI Scene Flow 2015 | ¿Objeto A más cercano o más lejano? |
| 101-200 | Ilusiones visuales de profundidad | 3D Visual Illusion | ¿Objeto A más cercano o más lejano? |

Cada bloque se subdivide en 4 condiciones (25 tareas cada una):
- Alta disparidad + sin distractores
- Alta disparidad + con distractores
- Baja disparidad + sin distractores
- Baja disparidad + con distractores

## 1. Configuración

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, subprocess, json, random, time
import torch
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

PROJECT_ROOT = '/content/drive/MyDrive/cognitive-depth-model'
REPO_DIR = '/content/cognitive-depth-model'
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'checkpoints')

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/yisusgoenaga/cognitive-depth-model.git {REPO_DIR}

result = subprocess.run(['find', REPO_DIR, '-name', 'cognitive_model.py'], capture_output=True, text=True)
model_path = result.stdout.strip()
src_dir = os.path.dirname(os.path.dirname(model_path))
sys.path.insert(0, src_dir)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

## 2. Cargar Modelo Entrenado

In [ ]:
from model.cognitive_model import CognitiveDepthModel

model = CognitiveDepthModel(
    in_channels=6, ngl_magno=8, ngl_parvo=8,
    v1_channels=16, v2_channels=32,
    v3_channels=64, v4_channels=64,
    v5_channels=128, integration_units=128, dropout=0.3,
).to(device)

best_path = os.path.join(CHECKPOINT_DIR, 'best_model_stage2.pth')
if not os.path.exists(best_path):
    best_path = os.path.join(CHECKPOINT_DIR, 'best_model.pth')

checkpoint = torch.load(best_path, weights_only=False, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f'Modelo cargado: {os.path.basename(best_path)}')

## 3. Preparar las 200 Tareas

### Bloque 1 (Tareas 1-100): Discriminación de profundidad relativa
Imágenes del conjunto de validación de KITTI.

### Bloque 2 (Tareas 101-200): Ilusiones visuales de profundidad
Imágenes del conjunto de validación del dataset de ilusiones.

In [ ]:
# === BLOQUE 1: Tareas KITTI (1-100) ===
KITTI_RAW = os.path.join(PROJECT_ROOT, 'data/raw/kitti')
SPLITS_FILE = os.path.join(PROJECT_ROOT, 'data/splits/kitti_splits.json')

with open(SPLITS_FILE) as f:
    splits = json.load(f)

# Usar el conjunto de validación de KITTI
kitti_val_ids = splits.get('kitti_validation', [])

# Si no hay suficientes en validación, usar test
if len(kitti_val_ids) < 100:
    print(f'Validación KITTI tiene {len(kitti_val_ids)} escenas, complementando con test...')
    kitti_val_ids = kitti_val_ids + splits.get('kitti_test', [])

# Rutas
TEST_LEFT = os.path.join(KITTI_RAW, 'data_scene_flow/testing/image_2')
TEST_RIGHT = os.path.join(KITTI_RAW, 'data_scene_flow/testing/image_3')
TRAIN_LEFT = os.path.join(KITTI_RAW, 'data_scene_flow/training/image_2')
TRAIN_RIGHT = os.path.join(KITTI_RAW, 'data_scene_flow/training/image_3')
TRAIN_DISP = os.path.join(KITTI_RAW, 'data_scene_flow/training/disp_occ_0')

# Construir lista de tareas KITTI
kitti_tasks = []

for sid in kitti_val_ids[:100]:
    # Buscar en training primero, luego en testing
    left_path = os.path.join(TRAIN_LEFT, f'{sid}.png')
    right_path = os.path.join(TRAIN_RIGHT, f'{sid}.png')
    disp_path = os.path.join(TRAIN_DISP, f'{sid}.png')

    if not os.path.exists(left_path):
        left_path = os.path.join(TEST_LEFT, f'{sid}.png')
        right_path = os.path.join(TEST_RIGHT, f'{sid}.png')
        disp_path = None  # Test set no tiene disparidad

    if os.path.exists(left_path):
        kitti_tasks.append({
            'scene_id': sid,
            'left_path': left_path,
            'right_path': right_path,
            'disp_path': disp_path if disp_path and os.path.exists(disp_path) else None,
            'source': 'KITTI',
        })

print(f'Tareas KITTI preparadas: {len(kitti_tasks)}')

# === BLOQUE 2: Tareas Ilusiones (101-200) ===
ILLUSION_PATH = os.path.join(PROJECT_ROOT, 'data/raw/illusions/images')
ILL_SPLITS_FILE = os.path.join(PROJECT_ROOT, 'data/splits/illusion_splits.json')

illusion_tasks = []

if os.path.exists(ILL_SPLITS_FILE):
    with open(ILL_SPLITS_FILE) as f:
        ill_splits = json.load(f)
    ill_val_ids = ill_splits.get('illusion_validation', [])
    if len(ill_val_ids) < 100:
        ill_val_ids = ill_val_ids + ill_splits.get('illusion_test', [])

    for sid in ill_val_ids[:100]:
        img_path = os.path.join(ILLUSION_PATH, f'{sid}.png')
        if os.path.exists(img_path):
            illusion_tasks.append({
                'scene_id': sid,
                'image_path': img_path,
                'source': 'Illusion',
            })
else:
    # Si no hay splits, usar las imágenes disponibles
    if os.path.exists(ILLUSION_PATH):
        all_ill = sorted(os.listdir(ILLUSION_PATH))[:100]
        for fname in all_ill:
            illusion_tasks.append({
                'scene_id': os.path.splitext(fname)[0],
                'image_path': os.path.join(ILLUSION_PATH, fname),
                'source': 'Illusion',
            })

print(f'Tareas Ilusiones preparadas: {len(illusion_tasks)}')
print(f'\nTotal de tareas: {len(kitti_tasks) + len(illusion_tasks)}')

In [ ]:
# Asignar condiciones experimentales a cada tarea
# Diseño factorial: disparidad (alta/baja) × distractores (con/sin)
# 4 condiciones × 25 tareas por condición = 100 tareas por bloque

conditions = [
    {'disparity': 'alta', 'distractors': 'sin'},
    {'disparity': 'alta', 'distractors': 'con'},
    {'disparity': 'baja', 'distractors': 'sin'},
    {'disparity': 'baja', 'distractors': 'con'},
]

def assign_conditions(tasks, n_per_condition=25):
    """Asigna condiciones experimentales balanceadas."""
    assigned = []
    for i, task in enumerate(tasks):
        condition_idx = i % 4
        task_with_condition = task.copy()
        task_with_condition['condition'] = conditions[condition_idx]
        task_with_condition['disparity_level'] = conditions[condition_idx]['disparity']
        task_with_condition['distractors'] = conditions[condition_idx]['distractors']
        task_with_condition['task_number'] = i + 1
        assigned.append(task_with_condition)
    return assigned

kitti_tasks_assigned = assign_conditions(kitti_tasks)
illusion_tasks_assigned = assign_conditions(illusion_tasks)

# Renumerar ilusiones desde 101
for i, task in enumerate(illusion_tasks_assigned):
    task['task_number'] = 101 + i

all_tasks = kitti_tasks_assigned + illusion_tasks_assigned

print(f'Tareas totales asignadas: {len(all_tasks)}')
print(f'\nDistribución de condiciones:')
for cond in conditions:
    count = sum(1 for t in all_tasks if t['condition'] == cond)
    print(f'  Disparidad {cond["disparity"]}, {cond["distractors"]} distractores: {count} tareas')

## 4. Ejecutar las 200 Tareas (3 Participantes Artificiales)

Según la tesis (sección 8.4.2.4), se realizan **3 ejecuciones completas**
para calcular variabilidad interna. Total: 600 respuestas.

In [ ]:
def load_stereo_pair_kitti(task, target_size=(64, 128)):
    """Carga un par estéreo de KITTI como tensor."""
    import cv2, numpy as np
    h, w = target_size
    img_l = cv2.imread(task['left_path'])
    img_r = cv2.imread(task['right_path'])
    if img_l is None or img_r is None:
        return None, None
    img_l = cv2.resize(img_l, (w, h)).astype(np.float32) / 255.0
    img_r = cv2.resize(img_r, (w, h)).astype(np.float32) / 255.0
    left_t = torch.from_numpy(img_l).permute(2, 0, 1)
    right_t = torch.from_numpy(img_r).permute(2, 0, 1)
    stereo = torch.cat([left_t, right_t], dim=0).unsqueeze(0)

    # Ground truth desde disparidad
    gt_label = None
    if task.get('disp_path'):
        disp = cv2.imread(task['disp_path'], cv2.IMREAD_UNCHANGED)
        if disp is not None:
            disp = cv2.resize(disp.astype(np.float32) / 256.0, (w, h))
            vl = disp[:, :w//2]; vl = vl[vl > 0]
            vr = disp[:, w//2:]; vr = vr[vr > 0]
            if len(vl) > 0 and len(vr) > 0:
                gt_label = 1.0 if vl.mean() > vr.mean() else 0.0
    return stereo, gt_label


def load_illusion_image(task, target_size=(64, 128)):
    """Carga una imagen de ilusión como par estéreo simulado."""
    import cv2, numpy as np
    h, w = target_size
    img = cv2.imread(task['image_path'])
    if img is None:
        return None, None
    img = cv2.resize(img, (w, h)).astype(np.float32) / 255.0
    # Para ilusiones: usar la misma imagen como ambos ojos
    # (la ilusión está en las claves monoculares, no en disparidad real)
    img_t = torch.from_numpy(img).permute(2, 0, 1)
    stereo = torch.cat([img_t, img_t], dim=0).unsqueeze(0)
    return stereo, None  # Sin ground truth para ilusiones


def run_single_task(model, task, device, run_id):
    """Ejecuta una tarea y registra respuesta + tiempo."""
    # Cargar imagen
    if task['source'] == 'KITTI':
        stereo, gt_label = load_stereo_pair_kitti(task)
    else:
        stereo, gt_label = load_illusion_image(task)

    if stereo is None:
        return None

    stereo = stereo.to(device)

    # Medir tiempo de procesamiento
    start_time = time.perf_counter()
    with torch.no_grad():
        output = model(stereo)
    end_time = time.perf_counter()

    prob = output.item()
    response = 'Más cercano' if prob >= 0.5 else 'Más lejano'
    response_binary = 1 if prob >= 0.5 else 0
    response_time_ms = (end_time - start_time) * 1000

    # Determinar si es correcto
    correct = None
    if gt_label is not None:
        correct = 1 if response_binary == int(gt_label) else 0

    return {
        'run_id': f'artificial_{run_id}',
        'task_number': task['task_number'],
        'scene_id': task['scene_id'],
        'source': task['source'],
        'task_type': 'Discriminación' if task['source'] == 'KITTI' else 'Ilusión',
        'disparity_level': task['disparity_level'],
        'distractors': task['distractors'],
        'response': response,
        'response_binary': response_binary,
        'probability': round(prob, 4),
        'response_time_ms': round(response_time_ms, 2),
        'ground_truth': gt_label,
        'correct': correct,
    }

print('Funciones de ejecución de tareas definidas.')

In [ ]:
# Ejecutar las 200 tareas × 3 participantes artificiales
NUM_RUNS = 3
all_results = []

print('=' * 60)
print('EJECUCIÓN DE 200 TAREAS × 3 PARTICIPANTES ARTIFICIALES')
print('=' * 60)

for run_id in range(1, NUM_RUNS + 1):
    print(f'\nParticipante artificial {run_id}/3...')
    run_results = []

    for i, task in enumerate(all_tasks):
        result = run_single_task(model, task, device, run_id)
        if result is not None:
            run_results.append(result)

        if (i + 1) % 50 == 0:
            print(f'  Completadas {i+1}/{len(all_tasks)} tareas...')

    all_results.extend(run_results)
    print(f'  Run {run_id}: {len(run_results)} respuestas registradas')

print(f'\nTotal de respuestas: {len(all_results)}')

## 5. Generar CSVs Estructurados

In [ ]:
# Crear DataFrame con todas las respuestas
results_df = pd.DataFrame(all_results)

# Guardar CSV principal
csv_path = os.path.join(PROJECT_ROOT, 'results/metrics/model_200_tasks_responses.csv')
results_df.to_csv(csv_path, index=False)

print(f'CSV principal guardado: {csv_path}')
print(f'Dimensiones: {results_df.shape}')
print(f'\nColumnas:')
for col in results_df.columns:
    print(f'  - {col}')
print(f'\nPrimeras 5 filas:')
results_df.head()

In [ ]:
# Resumen de desempeño por participante artificial
print('=' * 60)
print('DESEMPEÑO POR PARTICIPANTE ARTIFICIAL')
print('=' * 60)

for run_id in range(1, NUM_RUNS + 1):
    run_data = results_df[results_df['run_id'] == f'artificial_{run_id}']
    kitti_data = run_data[run_data['source'] == 'KITTI']
    ill_data = run_data[run_data['source'] == 'Illusion']

    # Precisión en KITTI (donde hay ground truth)
    kitti_valid = kitti_data[kitti_data['correct'].notna()]
    kitti_acc = kitti_valid['correct'].mean() if len(kitti_valid) > 0 else 0

    # Tiempos de respuesta
    mean_rt = run_data['response_time_ms'].mean()

    print(f'\nParticipante artificial_{run_id}:')
    print(f'  Tareas completadas: {len(run_data)}')
    print(f'  KITTI - Precisión: {kitti_acc:.3f} ({len(kitti_valid)} tareas con GT)')
    print(f'  Ilusiones - Respuestas: {len(ill_data)}')
    print(f'  Tiempo promedio: {mean_rt:.2f} ms')

    # Distribución de respuestas
    closer = (run_data['response_binary'] == 1).sum()
    farther = (run_data['response_binary'] == 0).sum()
    print(f'  Respuestas: {closer} más cercano, {farther} más lejano')

In [ ]:
# Resumen por condición experimental
print('=' * 60)
print('DESEMPEÑO POR CONDICIÓN EXPERIMENTAL')
print('=' * 60)

# Solo tareas KITTI (con ground truth)
kitti_results = results_df[(results_df['source'] == 'KITTI') & (results_df['correct'].notna())]

if len(kitti_results) > 0:
    condition_summary = kitti_results.groupby(['disparity_level', 'distractors']).agg(
        n_tasks=('correct', 'count'),
        accuracy=('correct', 'mean'),
        mean_rt=('response_time_ms', 'mean'),
        std_rt=('response_time_ms', 'std'),
    ).round(3)

    print(condition_summary)
    print()

    # Guardar resumen por condición
    cond_csv = os.path.join(PROJECT_ROOT, 'results/metrics/model_performance_by_condition.csv')
    condition_summary.to_csv(cond_csv)
    print(f'Guardado en: {cond_csv}')

In [ ]:
# Consistencia entre participantes artificiales (variabilidad interna)
print('=' * 60)
print('CONSISTENCIA ENTRE PARTICIPANTES ARTIFICIALES')
print('=' * 60)

# Comparar respuestas tarea por tarea entre los 3 runs
kitti_only = results_df[results_df['source'] == 'KITTI'].copy()

if len(kitti_only) > 0:
    # Pivotear: una fila por tarea, una columna por run
    pivot = kitti_only.pivot_table(
        index='task_number',
        columns='run_id',
        values='response_binary',
        aggfunc='first'
    )

    # Calcular acuerdo
    if pivot.shape[1] >= 2:
        agreement = (pivot.nunique(axis=1) == 1).mean()
        print(f'\nAcuerdo entre 3 runs: {agreement:.1%}')
        print(f'(Proporción de tareas con respuesta idéntica en los 3 runs)')

        # Nota: en un modelo determinístico sin dropout en eval,
        # el acuerdo debería ser ~100%
        if agreement > 0.99:
            print('\nNota: El acuerdo es ~100% porque el modelo en modo eval')
            print('es determinístico (dropout desactivado). Esto es esperado')
            print('y confirma la estabilidad del modelo.')

## 6. Visualizaciones

In [ ]:
# Gráfica: Precisión por condición
if len(kitti_results) > 0:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('Desempeño del Modelo en 200 Tareas de Validación',
                 fontsize=14, fontweight='bold')

    # 1. Precisión por condición
    cond_acc = kitti_results.groupby(['disparity_level', 'distractors'])['correct'].mean()
    labels = [f'{d}\n{dist} dist.' for (d, dist) in cond_acc.index]
    colors = ['#2196F3', '#FF9800', '#4CAF50', '#F44336']
    bars = axes[0].bar(labels, cond_acc.values, color=colors, alpha=0.8)
    axes[0].axhline(y=0.5, color='gray', linestyle='--', label='Azar')
    axes[0].set_ylabel('Precisión')
    axes[0].set_title('Precisión por Condición\n(KITTI)')
    axes[0].set_ylim(0, 1)
    axes[0].legend()
    for bar, val in zip(bars, cond_acc.values):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                     f'{val:.2f}', ha='center', fontweight='bold')

    # 2. Tiempos de respuesta por condición
    cond_rt = kitti_results.groupby(['disparity_level', 'distractors'])['response_time_ms'].mean()
    axes[1].bar(labels, cond_rt.values, color=colors, alpha=0.8)
    axes[1].set_ylabel('Tiempo de respuesta (ms)')
    axes[1].set_title('Tiempos de Respuesta por Condición\n(KITTI)')
    for bar, val in zip(axes[1].patches, cond_rt.values):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                     f'{val:.1f}', ha='center', fontsize=9)

    # 3. Distribución de probabilidades por tipo de tarea
    run1 = results_df[results_df['run_id'] == 'artificial_1']
    kitti_probs = run1[run1['source'] == 'KITTI']['probability']
    ill_probs = run1[run1['source'] == 'Illusion']['probability']
    axes[2].hist(kitti_probs, bins=15, alpha=0.6, label='KITTI', color='blue')
    axes[2].hist(ill_probs, bins=15, alpha=0.6, label='Ilusiones', color='orange')
    axes[2].axvline(x=0.5, color='black', linestyle='--')
    axes[2].set_xlabel('Probabilidad predicha')
    axes[2].set_ylabel('Frecuencia')
    axes[2].set_title('Distribución de Predicciones\npor Tipo de Tarea')
    axes[2].legend()

    plt.tight_layout()
    plt.savefig(os.path.join(PROJECT_ROOT, 'results/visualizations/validation_200_tasks.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print('Guardado en results/visualizations/validation_200_tasks.png')

In [ ]:
# Resumen final
print('=' * 60)
print('NOTEBOOK 07 COMPLETADO')
print('=' * 60)
print()
print(f'Tareas ejecutadas: {len(all_tasks)} × {NUM_RUNS} runs = {len(all_results)} respuestas')
print()
print('Archivos generados:')
print(f'  results/metrics/model_200_tasks_responses.csv')
print(f'  results/metrics/model_performance_by_condition.csv')
print(f'  results/visualizations/validation_200_tasks.png')
print()
print('Estos CSVs contienen los datos del modelo artificial')
print('listos para comparar con los datos de participantes humanos')
print('mediante el análisis estadístico descrito en la sección 8.5:')
print('  - GLMM binomial (precisión)')
print('  - ANOVA factorial (tiempos de respuesta)')
print('  - Chi-cuadrado (errores en ilusiones)')
print('  - Kappa de Cohen (concordancia ensayo a ensayo)')
print('  - Correlación de perfiles por condición')